# Análisis de sensibilidad de variantes NO-PASS en genes candidatos MVA

**MVA Hackathon 2026 · Sensibilidad del filtrado VCF**

Este notebook evalúa las variantes clasificadas como `NO-PASS` por el filtrado técnico del VCF en los 6 genes candidatos de MVA

El objetivo es determinar si el filtrado estricto de variantes `PASS` podría haber excluido variantes potencialmente relevantes para el caso clínico.

El análisis utiliza la misma lista de genes candidatos y las mismas coordenadas utilizadas en `02_filtrado_vcf_genes_mva.ipynb`, pero conserva únicamente las variantes cuyo campo `FILTER` es diferente de `PASS`.

In [1]:
import gzip
import json
import requests
import pandas as pd
from pathlib import Path

In [2]:
from pathlib import Path

DATA_DIR = Path("../data/raw")
RESULTS_DIR = Path("../results")
VCF = DATA_DIR / "WGS_EX2312012_HGWCNDSX7.vcf.gz"

RESULTS_DIR.mkdir(exist_ok=True)

print("VCF:", VCF)
print("Resultados:", RESULTS_DIR)

VCF: ../data/raw/WGS_EX2312012_HGWCNDSX7.vcf.gz
Resultados: ../results


In [3]:
FALLBACK_COORDS = {
    "BUB1B":  ("15", 40_160_984,  40_221_137),
    "CEP57":  ("11", 95_789_965,  95_837_070),
    "TRIP13": ("5",    892_849,      919_357),
    "BUB1":   ("2",  110_143_783, 110_244_769),
    "MAD1L1": ("7",   1_815_787,   2_233_243),
    "MAD2L1": ("4",  120_055_623, 120_067_192),
}

PADDING = 5_000

def fetch_ensembl(gene):
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?content-type=application/json"
    try:
        d = requests.get(url, timeout=10).json()
        return str(d["seq_region_name"]), int(d["start"]), int(d["end"])
    except Exception:
        return None

gene_coords = {}

for gene, fb in FALLBACK_COORDS.items():
    result = fetch_ensembl(gene)

    if result:
        gene_coords[gene] = result
        print(
            f"{gene:8s}  {result[0]}:{result[1]:>12,}-{result[2]:>12,}  (Ensembl)"
        )
    else:
        gene_coords[gene] = fb
        print(
            f"{gene:8s}  {fb[0]}:{fb[1]:>12,}-{fb[2]:>12,}  (fallback)"
        )

with open(RESULTS_DIR / "mva_gene_coords_grch38.json", "w") as f:
    json.dump(gene_coords, f, indent=2)

BUB1B     15:  40,160,984-  40,221,137  (Ensembl)
CEP57     11:  95,789,965-  95,837,070  (Ensembl)
TRIP13    5:     892,849-     919,357  (Ensembl)
BUB1      2: 110,635,468- 110,678,098  (Ensembl)
MAD1L1    7:   1,815,787-   2,233,243  (fallback)
MAD2L1    4: 120,055,623- 120,067,192  (fallback)


In [4]:
# Construir índice cromosoma → lista de intervalos de genes candidatos

interval_idx = {}

for gene, (chrom, start, end) in gene_coords.items():
    interval_idx.setdefault(chrom, []).append(
        (start - PADDING, end + PADDING, gene)
    )


def gene_at(chrom, pos):
    for start, end, gene in interval_idx.get(chrom, []):
        if start <= pos <= end:
            return gene
    return None

In [5]:
def classify_zygosity(gt):
    alleles = gt.replace("|", "/").split("/")

    if "." in alleles:
        return "MISSING"

    if set(alleles) == {"0"}:
        return "HOM_REF"

    if alleles[0] == alleles[1] and alleles[0] != "0":
        return "HOM_ALT"

    if "0" in alleles:
        return "HET"

    return "OTHER"

In [6]:
hits = []

print(f"Escaneando {VCF.name} ...")

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info, fmt = p[:9]
        sample = p[9] if len(p) > 9 else ""

        # Sensibilidad: conservar únicamente variantes NO-PASS
        if filt == "PASS":
            continue

        pos = int(pos_s)
        gene = gene_at(chrom, pos)

        if not gene:
            continue

        gt_dict = dict(zip(fmt.split(":"), sample.split(":")))

        gt = gt_dict.get("GT", ".")
        dp = int(gt_dict.get("DP", 0) or 0)
        gq = int(gt_dict.get("GQ", 0) or 0)
        ad = gt_dict.get("AD", ".")

        var_type = (
            "SNV" if len(ref) == 1 and len(alt) == 1
            else "DEL" if len(ref) > len(alt)
            else "INS"
        )

        hits.append({
            "gene": gene,
            "chrom": chrom,
            "pos": pos,
            "id": vid,
            "ref": ref,
            "alt": alt,
            "qual": float(qual),
            "filter": filt,
            "type": var_type,
            "zygosity": classify_zygosity(gt),
            "GT": gt,
            "DP": dp,
            "GQ": gq,
            "AD": ad,
        })

df_nonpass = pd.DataFrame(hits)

df_nonpass.to_csv(
    RESULTS_DIR / "variants_mva_genes_nonpass.tsv",
    sep="\t",
    index=False
)

print(f"Total variantes NO-PASS en genes candidatos: {len(df_nonpass):,}")

Escaneando WGS_EX2312012_HGWCNDSX7.vcf.gz ...
Total variantes NO-PASS en genes candidatos: 6


In [7]:
print("=== TODAS LAS VARIANTES NO-PASS EN GENES CANDIDATOS ===")
print(df_nonpass.to_string(index=False))

print("\n=== Variantes NO-PASS por gen ===")
print(
    df_nonpass.groupby("gene")[["pos"]]
    .count()
    .rename(columns={"pos": "total"})
    .sort_values("total", ascending=False)
)

print("\n=== Por gen y zigosidad ===")
print(pd.crosstab(df_nonpass["gene"], df_nonpass["zygosity"]))

print("\n=== Por tipo de variante ===")
print(
    df_nonpass.groupby(["gene", "type"])["pos"]
    .count()
    .unstack(fill_value=0)
)

print("\n=== Motivo(s) de NO-PASS ===")
print(df_nonpass["filter"].value_counts())

=== TODAS LAS VARIANTES NO-PASS EN GENES CANDIDATOS ===
  gene chrom      pos          id                                ref alt   qual      filter type zygosity  GT  DP  GQ    AD
MAD1L1     7  1827446           . TCCTCCCCTCCTGAGCCCGTCGCGGGTGTGGGGG   T  43.73         QD2  DEL      HET 0/1  27  81  23,4
MAD1L1     7  1832496  rs66617818                                  C   A 457.77        MQ40  SNV  HOM_ALT 1/1  14  42  0,14
MAD1L1     7  1877508 rs138324521                                  C   T 540.77        MQ40  SNV      HET 0/1  35  99 16,19
MAD1L1     7  1996262   rs6969969                                  G   A  21.80 LowQual;QD2  SNV      HET 0/1  15  50  13,2
MAD1L1     7  1996263   rs6969970                                  G   T  21.80 LowQual;QD2  SNV      HET 0/1  15  50  13,2
 BUB1B    15 40180642           .                                 CT   C  19.78 LowQual;QD2  DEL      HET 0/1  13  57   8,5

=== Variantes NO-PASS por gen ===
        total
gene         
MAD1L1      5

## Interpretación del análisis de sensibilidad

El filtrado principal del VCF conserva únicamente variantes `PASS`. Este análisis de sensibilidad evalúa las variantes `NO-PASS` localizadas en los genes candidatos de MVA para determinar si alguna variante potencialmente relevante podría haber sido excluida por los filtros técnicos.

Las variantes `NO-PASS` no se consideran patogénicas por este motivo. El objetivo es caracterizarlas según:

- gen afectado;
- tipo de variante;
- cigosidad;
- calidad de la variante (`QUAL`);
- profundidad de secuenciación (`DP`);
- calidad del genotipo (`GQ`);
- balance de alelos (`AD`);
- motivo por el cual la variante no fue clasificada como `PASS`.

Este análisis permite identificar variantes que podrían requerir una revisión adicional, especialmente en genes candidatos relevantes para el fenotipo.

In [8]:
# Mostrar los campos INFO de las variantes NO-PASS

print("=== INFO de las variantes NO-PASS ===")

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info = p[:8]

        if filt == "PASS":
            continue

        pos = int(pos_s)
        gene = gene_at(chrom, pos)

        if gene:
            print(
                f"{gene:7s} {chrom}:{pos} | "
                f"FILTER={filt} | "
                f"QUAL={qual} | "
                f"INFO={info}"
            )

=== INFO de las variantes NO-PASS ===
MAD1L1  7:1827446 | FILTER=QD2 | QUAL=43.73 | INFO=AC=1;AF=0.5;AN=2;BaseQRankSum=-0.509;ClippingRankSum=0.000;DP=27;ExcessHet=3.0103;FS=5.316;MLEAC=1;MLEAF=0.5;MQ=54.59;MQRankSum=-3.802;QD=1.62;ReadPosRankSum=0.499;SOR=1.492
MAD1L1  7:1832496 | FILTER=MQ40 | QUAL=457.77 | INFO=AC=2;AF=1;AN=2;DB;DP=14;ExcessHet=3.0103;FS=0.000;MLEAC=2;MLEAF=1;MQ=37.01;QD=32.70;SOR=0.976
MAD1L1  7:1877508 | FILTER=MQ40 | QUAL=540.77 | INFO=AC=1;AF=0.5;AN=2;BaseQRankSum=4.320;ClippingRankSum=0.000;DB;DP=35;ExcessHet=3.0103;FS=0.000;MLEAC=1;MLEAF=0.5;MQ=39.51;MQRankSum=0.282;QD=15.45;ReadPosRankSum=1.463;SOR=0.693
MAD1L1  7:1996262 | FILTER=LowQual;QD2 | QUAL=21.80 | INFO=AC=1;AF=0.5;AN=2;BaseQRankSum=1.852;ClippingRankSum=0.000;DB;DP=16;ExcessHet=3.0103;FS=3.222;MLEAC=1;MLEAF=0.5;MQ=44.46;MQRankSum=1.526;QD=1.45;ReadPosRankSum=2.368;SOR=0.124
MAD1L1  7:1996263 | FILTER=LowQual;QD2 | QUAL=21.80 | INFO=AC=1;AF=0.5;AN=2;BaseQRankSum=-0.674;ClippingRankSum=0.000;DB;DP=16;

In [9]:
# Resumen técnico de las variantes NO-PASS

summary_nonpass = df_nonpass[
    ["gene", "chrom", "pos", "ref", "alt",
     "qual", "filter", "zygosity", "GT", "DP", "GQ", "AD"]
].copy()

summary_nonpass

,gene,chrom,pos,ref,alt,qual,filter,zygosity,GT,DP,GQ,AD
0,MAD1L1,7,1827446,TCCTCCCCTCCTGAGCCCGTCGCGGGTGTGGGGG,T,43.73,QD2,HET,0/1,27,81,"23,4"
1,MAD1L1,7,1832496,C,A,457.77,MQ40,HOM_ALT,1/1,14,42,"0,14"
2,MAD1L1,7,1877508,C,T,540.77,MQ40,HET,0/1,35,99,"16,19"
3,MAD1L1,7,1996262,G,A,21.80,LowQual;QD2,HET,0/1,15,50,"13,2"
4,MAD1L1,7,1996263,G,T,21.80,LowQual;QD2,HET,0/1,15,50,"13,2"
5,BUB1B,15,40180642,CT,C,19.78,LowQual;QD2,HET,0/1,13,57,"8,5"


In [11]:
# Extraer métricas técnicas del campo INFO

def parse_info(info):
    values = {}

    for item in info.split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            values[key] = value

    return values


technical_hits = []

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info = p[:8]

        if filt == "PASS":
            continue

        pos = int(pos_s)
        gene = gene_at(chrom, pos)

        if not gene:
            continue

        info_dict = parse_info(info)

        technical_hits.append({
            "gene": gene,
            "chrom": chrom,
            "pos": pos,
            "id": vid,
            "ref": ref,
            "alt": alt,
            "QUAL": float(qual),
            "FILTER": filt,
            "DP": float(info_dict.get("DP", "nan")),
            "QD": float(info_dict.get("QD", "nan")),
            "MQ": float(info_dict.get("MQ", "nan")),
            "FS": float(info_dict.get("FS", "nan")),
            "MQRankSum": float(info_dict.get("MQRankSum", "nan")),
            "ReadPosRankSum": float(info_dict.get("ReadPosRankSum", "nan")),
            "SOR": float(info_dict.get("SOR", "nan")),
        })

df_technical = pd.DataFrame(technical_hits)

df_technical.reset_index(drop=True)

,gene,chrom,pos,id,ref,alt,QUAL,FILTER,DP,QD,MQ,FS,MQRankSum,ReadPosRankSum,SOR
0,MAD1L1,7,1827446,.,TCCTCCCCTCCTGAGCCCGTCGCGGGTGTGGGGG,T,43.73,QD2,27.0,1.62,54.59,5.316,-3.802,0.499,1.492
1,MAD1L1,7,1832496,rs66617818,C,A,457.77,MQ40,14.0,32.70,37.01,0.000,NaN,NaN,0.976
2,MAD1L1,7,1877508,rs138324521,C,T,540.77,MQ40,35.0,15.45,39.51,0.000,0.282,1.463,0.693
3,MAD1L1,7,1996262,rs6969969,G,A,21.80,LowQual;QD2,16.0,1.45,44.46,3.222,1.526,2.368,0.124
4,MAD1L1,7,1996263,rs6969970,G,T,21.80,LowQual;QD2,16.0,1.45,44.46,3.222,1.526,2.417,0.124
5,BUB1B,15,40180642,.,CT,C,19.78,LowQual;QD2,16.0,1.52,60.00,8.921,-0.000,-0.428,0.022


## Conclusión (A9)

Ninguna de las 6 variantes `NO-PASS` cambia el diagnóstico. Separadas en dos grupos según el motivo del filtro:

- **`MQ40` (7:1832496, 7:1877508, ambas en *MAD1L1*):** `QUAL` alto (457.77 y 540.77), `GQ` 42 y 99, buen soporte de lecturas alternativas (una incluso homocigota, 14/14; la otra 19/35 ≈ 54 %) y ambas con rsID de dbSNP (`rs66617818`, `rs138324521`). Todo indica que son variantes **reales y probablemente comunes**, descartadas solo por mapeo ambiguo (`MQ` 37–40, justo en el corte), no por ser falsas.
- **`QD2`/`LowQual` (7:1827446 y 7:1996262/63 en *MAD1L1*; 15:40180642 en *BUB1B*):** `QUAL` bajo (19.8–43.7), `QD` ≤ 1.6 y muy poco soporte alternativo (4/27, 2/15 dos veces, 5/13). Son compatibles con **artefactos técnicos** de baja profundidad o de una región difícil de alinear (la de *MAD1L1* cae en una secuencia rica en G al final del *indel*), no con variantes reales.

La variante `NO-PASS` de *BUB1B* (15:40180642) es, de las 6, la de menor calidad (`QUAL` 19.78, `DP` 13): no hay motivo para tratarla como un tercer evento que compita con `p.Leu737Ter` y `p.Asn1002Lys`. Tampoco se repitió la anotación funcional (VEP/MANE) de ninguna de las 6, porque la calidad técnica ya alcanza para descartarlas antes de llegar a esa pregunta.

*MAD1L1* no tiene patrón recesivo entre sus variantes `PASS` (ver `scripts/diagnostico_diferencial.py`), así que aun si sus 4 variantes `NO-PASS` fueran reales, no cambiarían la conclusión de que *BUB1B* es la causa.

**En síntesis:** el filtro `PASS` del pipeline no dejó afuera ninguna variante que debiera revisarse para este caso.

> Borrador de Ktalyna (con Claude Code), 8 oct — pendiente de que Josefina lo confirme y lo firme como su conclusión (tarea A9).